In [4]:
!pip install langchain langchain-community langchain-google-genai


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain.tools import tool

In [6]:
import os 
# google AI studio
os.environ["GOOGLE_API_KEY"] = "AQ.Ab8RN6IOjft_ZllZ5hw37ZQiM2i9uE0TWycKjHUrgEP2Q6cOZA"

In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=1
)

In [8]:
!pip install pypdf


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
!pip install -U langchain-community langchain-google-genai langchain


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
from pathlib import Path
from langchain.tools import tool
from langchain_community.document_loaders import PyPDFLoader

In [29]:
PDF_FILE = Path(r"sample_placement_data.pdf")

loader = PyPDFLoader(str(PDF_FILE))
documents = loader.load()

# This is the missing variable that caused your NameError.
pdf_text = "\n".join(doc.page_content for doc in documents)

print(f"PDF loaded successfully: {len(documents)} pages")
print(f"Extracted text: {len(pdf_text)} characters")


@tool
def read_pdf(query: str) -> str:
    """Search the placement PDF using a query."""
    matches = [
        line.strip()
        for line in pdf_text.splitlines()
        if query.lower() in line.lower()
    ]

    if matches:
        return "\n".join(matches[:30])

    return "No relevant information found in the placement PDF."


PDF loaded successfully: 3 pages
Extracted text: 4836 characters


In [30]:
@tool
def search_placement_info(query: str) -> str:
    """
    Search the placement PDF for information about companies, roles,
    packages, deadlines, branches, CGPA, or selection processes.
    """
    query_words = query.lower().split()
    matches = []

    for line in pdf_text.splitlines():
        line = line.strip()

        if line and any(word in line.lower() for word in query_words):
            matches.append(line)

    if matches:
        return "\n".join(matches[:25])

    return "No matching placement information was found in the PDF."

In [31]:
@tool
def get_company_details(company_name : str) -> str:
    """Get Information of the Company"""
    result = []
    for line in pdf_text.split("\n"):
        if company_name.lower() in line.lower():
            result.append(line)
    if result:
        return "\n".join(result)
    return f"No information found for {company_name}."

In [32]:
@tool
def check_eligibility(
    company_name: str,
    cgpa: float,
    branch: str,
    active_backlogs: int
) -> str:
    """
    Check whether a student may be eligible for a company.
    Provide company name, CGPA, branch, and active backlog count.
    """
    company_info = get_company_details.invoke({
        "company_name": company_name
    })

    return (
        f"Student details:\n"
        f"Company: {company_name}\n"
        f"CGPA: {cgpa}\n"
        f"Branch: {branch}\n"
        f"Active backlogs: {active_backlogs}\n\n"
        f"Company information from PDF:\n{company_info}\n\n"
        f"Compare the student's CGPA, branch, and backlog count with "
        f"the eligibility rules above. Clearly state Eligible, Not Eligible, "
        f"or Needs Placement Cell Confirmation."
    )

In [33]:
agent = create_agent(
    model=llm,
    tools=[
        search_placement_info,
        get_company_details,
        check_eligibility
    ],
    system_prompt="""
You are a college placement assistant.

Use the available tools whenever the user asks about:
- companies
- jobs or roles
- salary packages
- eligibility
- CGPA
- branches
- backlogs
- deadlines
- selection processes
- general placement rules

Use only information from the placement PDF tools.
If the information is unavailable, say so clearly.
"""
)

In [ ]:
question = input("Ask a placement question: ")

result = agent.invoke({
    "messages": [
        {"role": "user", "content": question}
    ]
})

answer = result["messages"][-1].content

if isinstance(answer, list):
    answer = "\n".join(
        item["text"]
        for item in answer
        if isinstance(item, dict) and item.get("type") == "text"
    )

print("\n" + "=" * 60)
print("Question:", question)
print("=" * 60)
print("Answer:", answer)
print("=" * 60)

Ask a placement question:  Google


C:\Users\Danish Basha\AppData\Local\Programs\Python\Python313\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
C:\Users\Danish Basha\AppData\Local\Programs\Python\Python313\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
